# Track Ordering Analysis

Check whether tornado tracks in the .pt files have any inherent ordering (temporal, spatial, etc.) by plotting start points colored by their array index.

In [ ]:
!pip install -q --no-cache-dir git+https://github.com/jcaramichaellehigh/TorGen.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import torch
import os
import matplotlib.pyplot as plt
import numpy as np

RAW_DIR = "/content/drive/MyDrive/raw"

def load_day(date_str):
    """Load a .pt file by date string."""
    path = os.path.join(RAW_DIR, date_str + ".pt")
    return torch.load(path, weights_only=False)

# Big outbreak days
dates = ["2011-04-27", "2023-03-31", "2012-04-14"]
print("Loading...")
days = {d: load_day(d) for d in dates}
for d, data in days.items():
    print(f"{d}: {data['tracks'].shape[0]} tracks")

In [ ]:
fig, axes = plt.subplots(1, len(dates), figsize=(6 * len(dates), 6))
if len(dates) == 1:
    axes = [axes]

for ax, date_str in zip(axes, dates):
    tracks = days[date_str]["tracks"]  # (N, 6): se, sn, ee, en, width, ef
    n = tracks.shape[0]
    se = tracks[:, 0].numpy()
    sn = tracks[:, 1].numpy()

    # Color by array index (order in file)
    colors = np.arange(n)
    sc = ax.scatter(se, sn, c=colors, cmap="viridis", s=15, alpha=0.8)

    # Annotate first and last few with their index
    for i in [0, 1, 2, n - 3, n - 2, n - 1]:
        if i < n:
            ax.annotate(str(i), (se[i], sn[i]), fontsize=7, color="red")

    ax.set_xlabel("Start Easting (se)")
    ax.set_ylabel("Start Northing (sn)")
    ax.set_title(f"{date_str} ({n} tracks)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    plt.colorbar(sc, ax=ax, label="Array index (order in file)")

fig.suptitle("Track start points colored by array index", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Check if there's a monotonic trend in any coordinate vs index
from scipy.stats import spearmanr

print(f"{'Date':<14} {'N':>5}  {'r(idx,se)':>10} {'r(idx,sn)':>10} {'r(idx,ee)':>10} {'r(idx,en)':>10}")
print("-" * 65)
for date_str in dates:
    tracks = days[date_str]["tracks"]
    n = tracks.shape[0]
    idx = np.arange(n)
    cols = {"se": 0, "sn": 1, "ee": 2, "en": 3}
    corrs = {}
    for name, col in cols.items():
        r, _ = spearmanr(idx, tracks[:, col].numpy())
        corrs[name] = r
    print(f"{date_str:<14} {n:>5}  {corrs['se']:>10.3f} {corrs['sn']:>10.3f} {corrs['ee']:>10.3f} {corrs['en']:>10.3f}")